In [1]:
# convert.py
from PIL import Image
import os

def rgb565_to_hex(r, g, b):
    """将RGB颜色转换为RGB565格式"""
    return ((r >> 3) << 11) | ((g >> 2) << 5) | (b >> 3)

def convert_image_to_c_array(input_path, output_path, frame_num):
    """将单张图片转换为C数组"""
    # 打开图片并调整到72x78
    img = Image.open(input_path).resize((72, 78))
    
    # 转换为RGB模式（如果是RGBA需要处理透明）
    if img.mode == 'RGBA':
        # 用白色背景填充透明部分
        background = Image.new('RGB', img.size, (255, 255, 255))
        background.paste(img, mask=img.split()[3])
        img = background
    else:
        img = img.convert('RGB')
    
    # 提取像素数据（水平扫描，从上到下，从左到右）
    pixels = []
    for y in range(img.height):
        for x in range(img.width):
            r, g, b = img.getpixel((x, y))
            pixels.append(rgb565_to_hex(r, g, b))
    
    # 生成C数组
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(f'// 帧{frame_num}: 72x78 RGB565\n')
        f.write(f'const uint16_t frame{frame_num}[5616] = {{\n')
        
        # 每行写12个数据，方便阅读
        for i in range(0, len(pixels), 12):
            line = '    ' + ', '.join([f'0x{p:04X}' for p in pixels[i:i+12]])
            if i + 12 < len(pixels):
                line += ','
            f.write(line + '\n')
        
        f.write('};\n\n')

# 批量转换
def batch_convert():
    # 获取所有图片文件（支持常见格式）
    image_extensions = ('.png')#, '.jpg', '.jpeg', '.bmp', '.gif'
    image_files = [f for f in os.listdir('.') if f.lower().endswith(image_extensions)]
    
    # 按文件名排序
    image_files.sort()
    
    if not image_files:
        print('错误：当前文件夹没有找到图片文件！')
        return
    
    print(f'找到 {len(image_files)} 张图片，开始转换...')
    
    # 生成所有帧的C文件
    all_frames = []
    for idx, img_file in enumerate(image_files, 1):
        output_file = f'frame{idx}.c'
        convert_image_to_c_array(img_file, output_file, idx)
        all_frames.append(f'frame{idx}.c')
        print(f'✅ 已转换: {img_file} -> {output_file}')
    
    # 生成汇总头文件
    with open('frames.h', 'w', encoding='utf-8') as f:
        f.write('#ifndef __FRAMES_H\n')
        f.write('#define __FRAMES_H\n\n')
        f.write('#include <stdint.h>\n\n')
        f.write('#define FRAME_W  72\n')
        f.write('#define FRAME_H  78\n')
        f.write(f'#define FRAME_COUNT  {len(all_frames)}\n\n')
        
        for idx in range(1, len(all_frames) + 1):
            f.write(f'extern const uint16_t frame{idx}[5616];\n')
        
        f.write('\n// 帧指针数组\n')
        f.write(f'extern const uint16_t *frames[{len(all_frames)}];\n\n')
        f.write('#endif\n')
    
    # 生成汇总C文件
    with open('frames.c', 'w', encoding='utf-8') as f:
        f.write('#include "frames.h"\n\n')
        
        for idx in range(1, len(all_frames) + 1):
            f.write(f'#include "frame{idx}.c"\n')
        
        f.write('\nconst uint16_t *frames[FRAME_COUNT] = {\n    ')
        f.write(', '.join([f'frame{i}' for i in range(1, len(all_frames) + 1)]))
        f.write('\n};\n')
    
    print(f'\n✅ 全部完成！共生成 {len(all_frames)} 个帧文件')
    print('📁 生成的文件：')
    for f in all_frames:
        print(f'   - {f}')
    print('   - frames.h')
    print('   - frames.c')

if __name__ == '__main__':
    batch_convert()

找到 28 张图片，开始转换...
✅ 已转换: 臭_0000_图层-28.png -> frame1.c
✅ 已转换: 臭_0001_图层-27.png -> frame2.c
✅ 已转换: 臭_0002_图层-26.png -> frame3.c
✅ 已转换: 臭_0003_图层-25.png -> frame4.c
✅ 已转换: 臭_0004_图层-24.png -> frame5.c
✅ 已转换: 臭_0005_图层-23.png -> frame6.c
✅ 已转换: 臭_0006_图层-22.png -> frame7.c
✅ 已转换: 臭_0007_图层-21.png -> frame8.c
✅ 已转换: 臭_0008_图层-20.png -> frame9.c
✅ 已转换: 臭_0009_图层-19.png -> frame10.c
✅ 已转换: 臭_0010_图层-18.png -> frame11.c
✅ 已转换: 臭_0011_图层-17.png -> frame12.c
✅ 已转换: 臭_0012_图层-16.png -> frame13.c
✅ 已转换: 臭_0013_图层-15.png -> frame14.c
✅ 已转换: 臭_0014_图层-14.png -> frame15.c
✅ 已转换: 臭_0015_图层-13.png -> frame16.c
✅ 已转换: 臭_0016_图层-12.png -> frame17.c
✅ 已转换: 臭_0017_图层-11.png -> frame18.c
✅ 已转换: 臭_0018_图层-10.png -> frame19.c
✅ 已转换: 臭_0019_图层-9.png -> frame20.c
✅ 已转换: 臭_0020_图层-8.png -> frame21.c
✅ 已转换: 臭_0021_图层-7.png -> frame22.c
✅ 已转换: 臭_0022_图层-6.png -> frame23.c
✅ 已转换: 臭_0023_图层-5.png -> frame24.c
✅ 已转换: 臭_0024_图层-4.png -> frame25.c
✅ 已转换: 臭_0025_图层-3.png -> frame26.c
✅ 已转换: 臭_0026_图层-2.png -> frame27.c
